In [148]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [149]:
import pandas as pd 
import numpy as np
import re

In [150]:
data = pd.read_excel(r'C:\Users\KS\Desktop\K4รายงานสต็อกการ์ด พร้อมทุน.xls' ,engine='calamine',header=11,usecols="A:F",dtype={'Unnamed: 3': str})
data.rename(columns={'Unnamed: 0':'DATE','Unnamed: 1':'Bill','เพิ่ม ':'details','Unnamed: 3':'value','ลด ':'sale','คงเหลือ ':'balance'} ,inplace=True)

In [151]:
# DATE == รหัสสินค้า มีชื่อสินค้าอยู่ ถ้าเป็นค่าว่างให้ดึงชื่อจากแถวก่อนหน้า
data['product_id'] = data.loc[data['DATE'] =='รหัสสินค้า', 'sale']
data['product_id'] = data['product_id'].ffill()

In [152]:
# DATE == คลัง มีชื่อสินค้าอยู่ ถ้าเป็นค่าว่างให้ดึงชื่อจากแถวก่อนหน้า
data['unit'] = data.loc[data['DATE'].astype(str).str.strip() == 'คลัง', 'balance']
data['unit'] = data['unit'].ffill()
data['unit'] = data['unit'].str.extract(r'(\d+)').fillna(0).astype(int)

In [153]:
# แปลงตรงๆ โดยบอกสไตล์ปฏิทินสากลไปก่อน
data['DATE'] = pd.to_datetime(data['DATE'], format='%d/%m/%Y', errors='coerce')

# ลบปีออก 543 ปี (ใช้ DateOffset)
data['DATE'] = data['DATE'] - pd.DateOffset(years=543)
data['DATE'] = pd.to_datetime(data['DATE'])

In [154]:
data.dropna(subset=['DATE'], inplace=True)

แปลงคงเหลือให้เป็นหน่วนเล็กสุด

In [155]:
import numpy as np
import pandas as pd


def parse_pack_piece(series):
    """ฟังก์ชันแยก (หน้าจุด, หลังจุด) จากรูปแบบ Pack.Piece

    เงื่อนไข:
    - ตัวหน้า: เห็นเลขไหนเอาเลขนั้น
    - ตัวหลัง: ข้าม 0 ตัวแรกๆ หลังจุดไปจนกว่าจะเจอ 1-9 พอเจอแล้ว เอานับตั้งแต่จุดนั้นไปจนจบสตริง
    """

    # 1. แปลงเป็น String แบบสะอาดดั้งเดิมที่สุด
    s_str = series.fillna("0").astype(str).str.strip()
    s_str = s_str.replace(["nan", "None", ""], "0")

    # 2. ดึงตัวเลขหน้าจุด
    front_str = s_str.str.split(".").str[0]
    front = pd.to_numeric(front_str, errors="coerce").fillna(0).astype(int)

    # 3. ดึงส่วนหลังจุด
    back_part = s_str.str.split(".").str[1].fillna("")

    # 4. Regex: ^0*(.*) -> ข้าม 0 ด้านหน้า แล้วจับตั้งแต่เลขที่ไม่ใช่ 0 ไปจนจบ
    extracted_back = back_part.str.extract(r"^0*(.*)")[0].fillna("")

    # 5. แก้จุดนี้: ใช้ .replace() บน Pandas Series แทน np.where เพื่อป้องการเปลี่ยนเป็น ndarray
    extracted_back = extracted_back.replace("", "0")

    # 6. แปลงส่วนหลังเป็น integer
    back = pd.to_numeric(extracted_back, errors="coerce").fillna(0).astype(int)

    return front, back

In [156]:
# แยกหน่วย Import (Value)
data['front_value'], data['back_value'] = parse_pack_piece(data['value'])

# แยกหน่วย Export (Sale)
data['front_sale'], data['back_sale'] = parse_pack_piece(data['sale'])

# แยกหน่วย Balances
data['front_balance'], data['back_balance'] = parse_pack_piece(data['balance'])

In [157]:
data['import'] = data['back_value'] + (data['unit'] * data['front_value']) 
data['export'] = data['back_sale'] + (data['unit'] * data['front_sale']) 
data['balances'] = data['back_balance'] + (data['unit'] * data['front_balance']) 

## Hybrid Robust Z-score (Global + Rolling) พร้อมคำนวณส่วนต่างและช่วงเวลาบิลหาย

**สิ่งที่เพิ่มเข้ามาใน V3:**
- **คอลัมน์ตรวจสอบบิลย้อนหลัง (Actionable Columns):** สำหรับเคส `🔴 บิลหายทั้งใบ` ระบบจะระบุวันและช่วงวันที่พนักงานบัญชี/คลังสินค้าต้องไปตามหาเอกสารทันที:
  1. `suspect_start_date` (วันที่เริ่มน่าสงสัย) = วันที่ของเข้าล่าสุด + 1 วัน
  2. `suspect_end_date` (วันที่สิ้นสุดการสงสัย) = วันที่ทำรายการเบิกออก/ขายปัจจุบัน
  3. `suspect_date_range` (ช่วงเวลาที่ต้องไปย้อนดูบิล) = รวมเป็นข้อความให้อ่านง่าย เช่น `2026-07-01 ถึง 2026-08-05`

In [ ]:
#%pip install scipy
#%pip install openpyxl
#%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ------------------------------------ --- 7.6/8.3 MB 52.9 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 44.2 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---


[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [159]:
import math
import re
import numpy as np
import openpyxl
import pandas as pd

# ==========================================
# 1. เตรียมข้อมูลและสร้าง Transaction Flags
# ==========================================
df = data[
    [
        'DATE',
        'Bill',
        'details',
        'product_id',
        'import',
        'export',
        'balances',
    ]
].copy()

df.columns = df.columns.str.strip()
df['product_id'] = df['product_id'].astype(str).str.strip()
df['details'] = df['details'].astype(str).str.strip()
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(by=['product_id', 'DATE']).reset_index(drop=True)

max_date_in_dataset = df['DATE'].max()

# Flag บิลรับคืน / NV
exclude_keywords = ['รับคืน', 'NV']
pattern_exclude = '|'.join(exclude_keywords)
df['is_return_bill'] = df['details'].str.contains(
    pattern_exclude, na=False, case=False
)
df['net_export'] = np.where(
    df['is_return_bill'], -df['import'], df['export']
)


# ==========================================
# 2. Adaptive Reference Calculation (Mode vs Median Dynamic Selection)
# ==========================================
import_history = df[(df['import'] > 0) & (~df['is_return_bill'])].copy()


def get_clean_frequent_import(group):
    import_series = group['import']
    import_series = import_series[import_series > 0]

    # 1. กรณีไม่มีประวัติรับเข้าเลย
    if import_series.empty:
        return 1.0, 1.0, 1.0  # mode, median, selected

    # 2. คำนวณ Mode และ Median
    modes = import_series.mode()
    has_mode = not modes.empty
    mode_val = modes.iloc[0] if has_mode else import_series.median()

    # ตัด Outlier ฝั่งสูงเกิน 3 เท่าออกชั่วคราวเพื่อหา Median ที่สะอาดขึ้น
    clean_series = import_series[import_series <= (mode_val * 3.0)]
    median_val = (
        clean_series.median() if not clean_series.empty else import_series.median()
    )

    # 3. 🎯 Dynamic Selection Logic (เลือกค่าที่เหมาะสมที่สุด)
    mode_count = (import_series == mode_val).sum()
    mode_ratio = mode_count / len(import_series)

    # ถ้า Mode เด่นชัด (ถี่เกิน 30% หรือซ้ำมากกว่า 1 ครั้ง) -> ใช้ Mode
    # ถ้าไม่มี Mode เด่นชัด (สั่งคละยอด/ไม่มีลังแน่นอน) -> ใช้ Median
    if has_mode and (mode_ratio >= 0.3 or mode_count > 1):
        best_val = mode_val
    else:
        best_val = median_val

    return mode_val, median_val, best_val


# คำนวณแบบปลอดภัย ป้องกัน Deprecation Warning
stats_list = []
for pid, group in import_history.groupby('product_id'):
    m_val, med_val, best_val = get_clean_frequent_import(group)
    stats_list.append(
        {
            'product_id': pid,
            'mode_import_size': m_val,
            'median_import_size': med_val,
            'selected_import_size': best_val,  # 👈 ค่าที่ระบบเลือกให้อัตโนมัติ
        }
    )

stats_df = pd.DataFrame(stats_list)

if not stats_df.empty:
    mode_map = stats_df.set_index('product_id')['mode_import_size'].to_dict()
    median_map = stats_df.set_index('product_id')['median_import_size'].to_dict()
    selected_map = stats_df.set_index('product_id')['selected_import_size'].to_dict()
else:
    mode_map, median_map, selected_map = {}, {}, {}

df['mode_import_size'] = df['product_id'].map(mode_map).fillna(df['import'])
df['median_import_size'] = df['product_id'].map(median_map).fillna(
    df['import']
)
df['selected_import_size'] = df['product_id'].map(selected_map).fillna(
    df['import']
)

# 🎯 ปรับจูนความไวได้ง่ายๆ แค่เปลี่ยนตัวเลขคูณ (เช่น 2.0, 2.5, 3.0)
df['is_outlier_by_mode'] = (
    (df['import'] > 0)
    & (df['selected_import_size'] > 1)
    & (df['import'] >= (df['selected_import_size'] * 2.5))  # 👈 ปรับ Sensitivity ตรงนี้
)


df['is_outlier_import'] = df['is_outlier_by_mode']


# ==========================================
# 3. คำนวณช่วงเวลาขาย (Idle Days) & Ghost / Dead Stock Flags
# ==========================================
sales_events = df[df['export'] > 0].copy()

if not sales_events.empty:
    sales_events['prev_export_date'] = sales_events.groupby('product_id')[
        'DATE'
    ].shift(1)
    sales_events['inter_sale_days'] = (
        sales_events['DATE'] - sales_events['prev_export_date']
    ).dt.days

    sale_stats = (
        sales_events.groupby('product_id')['inter_sale_days']
        .median()
        .reset_index()
    )
    sale_stats.rename(
        columns={'inter_sale_days': 'median_inter_sale_days'}, inplace=True
    )
    df = df.merge(sale_stats, on='product_id', how='left')
else:
    df['median_inter_sale_days'] = np.nan

df['median_inter_sale_days'] = df['median_inter_sale_days'].fillna(2.0)
df['median_inter_sale_days'] = np.maximum(df['median_inter_sale_days'], 1.0)

df['export_date_temp'] = df['DATE'].where(df['export'] > 0)
df['last_export_date'] = df.groupby('product_id')['export_date_temp'].ffill()

df['days_idle'] = (df['DATE'] - df['last_export_date']).dt.days.fillna(0)
df.drop(columns=['export_date_temp'], inplace=True)

df['current_idle_days_from_max'] = (
    max_date_in_dataset - df['last_export_date']
).dt.days.fillna(0)

# เช็กรายการขายถัดไปภายใน 3 วัน
df_rev = df.iloc[::-1].set_index('DATE')
df['next_3d_export'] = (
    df_rev.groupby('product_id')['export']
    .rolling('3D', min_periods=1)
    .sum()
    .iloc[::-1]
    .values
    - df['export']
)

# Flag Active Ghost Stock
df['dynamic_idle_threshold_c1'] = np.ceil(df['median_inter_sale_days'] * 1.5)
df['is_suspected_ghost'] = (
    (df['days_idle'] > df['dynamic_idle_threshold_c1'])
    & (df['balances'] > 0)
    & (df['import'] > 0)
    & (df['next_3d_export'] > 0)
)

# Flag Dead Stock
df['dynamic_idle_threshold_c2'] = np.ceil(df['median_inter_sale_days'] * 1.8)
df['is_last_record_per_sku'] = df.groupby('product_id')['DATE'].transform(
    'max'
) == df['DATE']

df['is_dead_last_item'] = (
    (df['is_last_record_per_sku'] == True)
    & (df['current_idle_days_from_max'] > df['dynamic_idle_threshold_c2'])
    & (df['balances'] > 0)
    & (df['import'] == 0)
)


# ==========================================
# 4. Reverse Stock Reconciliation & คำนวณ adjusted_calc_balance
# ==========================================
df['net_flow'] = df['import'] - df['export']

# ดึงยอดคงเหลือปัจจุบัน ณ บิลล่าสุด
last_balance_map = df.groupby('product_id')['balances'].last().to_dict()
df['current_last_balance'] = df['product_id'].map(last_balance_map)

# คำนวณ Expected_Import ตาม Priority
conditions_expected = [
    (df['is_suspected_ghost'] == True) & (df['import'] > 0),  # Ghost Stock
    (df['is_outlier_import'] == True),  # Over-Import
]

choices_expected = [
    np.maximum(df['import'] - df['current_last_balance'], 0.0),  # Reverse Logic
    df['selected_import_size'],  # 👈 ใช้ค่าที่ผ่านการเลือกอัตโนมัติ (Adaptive)
]

df['Expected_Import'] = np.select(
    conditions_expected, choices_expected, default=df['import']
)

df['Diff_Import'] = np.where(
    df['import'] > 0, df['import'] - df['Expected_Import'], np.nan
)

# คำนวณ adjusted_calc_balance
df['adjusted_import'] = df['Expected_Import']
df['adjusted_net_flow'] = df['adjusted_import'] - df['export']

first_balance_actual = df.groupby('product_id')['balances'].transform('first')
first_net_flow_actual = df.groupby('product_id')['net_flow'].transform('first')
true_initial_balance = first_balance_actual - first_net_flow_actual

df['adjusted_calc_balance'] = (
    true_initial_balance
    + df.groupby('product_id')['adjusted_net_flow'].cumsum()
)


# ==========================================
# 5. สรุปประเภท Anomaly
# ==========================================
df['is_stock_out'] = (df['balances'] <= 0) & (df['export'] > 0)
df['is_ghost_stock'] = (
    (df['is_suspected_ghost'] == True)
    & (df['adjusted_calc_balance'] <= 0)
    & (df['balances'] > 0)
)

anomaly_conditions = [
    (df['is_stock_out'] == True),
    (df['is_ghost_stock'] == True),
    (df['is_outlier_import'] == True),
    (df['is_dead_last_item'] == True),
]

anomaly_labels = [
    '🛑 สต็อกหมด/ติดลบ (Stock Out Alert)',
    '🚨 สต็อกผี/บิลเข้าคีย์เกินจนสต็อกบวม (Ghost Stock Alert)',
    '🔵 บิลรับเข้าพุ่งสูงผิดปกติ (Over-Import Anomaly)',
    '📦 สต็อกค้างนานไร้การเคลื่อนไหว (Dead Stock / 1.8x Idle)',
]

df['Anomaly_Type'] = np.select(
    anomaly_conditions, anomaly_labels, default='⚪ ปกติ (Normal)'
)

In [160]:
df.columns

Index(['DATE', 'Bill', 'details', 'product_id', 'import', 'export', 'balances',
       'is_return_bill', 'net_export', 'mode_import_size',
       'median_import_size', 'selected_import_size', 'is_outlier_by_mode',
       'is_outlier_import', 'median_inter_sale_days', 'last_export_date',
       'days_idle', 'current_idle_days_from_max', 'next_3d_export',
       'dynamic_idle_threshold_c1', 'is_suspected_ghost',
       'dynamic_idle_threshold_c2', 'is_last_record_per_sku',
       'is_dead_last_item', 'net_flow', 'current_last_balance',
       'Expected_Import', 'Diff_Import', 'adjusted_import',
       'adjusted_net_flow', 'adjusted_calc_balance', 'is_stock_out',
       'is_ghost_stock', 'Anomaly_Type'],
      dtype='str')

In [161]:
df[(df['product_id'] == 'ยิลเลตต์ธินทู 1ด้าม  แผง24') & (df['Bill'] == 'DM04256907/014')][['DATE', 'Bill', 'details', 'product_id', 'import', 'export', 'balances','mode_import_size','median_import_size','Expected_Import']]

,DATE,Bill,details,product_id,import,export,balances,mode_import_size,median_import_size,Expected_Import
61403,2026-07-14,DM04256907/014,โอนสินค้าสนง.ใหญ่ ไป - Kmart4 รับย้ายจาก.00,ยิลเลตต์ธินทู 1ด้าม แผง24,24,0,24,1.0,1.0,6.0


จับคู่ชื่อ

In [162]:
#%pip install rapidfuzz

In [163]:
search_terms = pd.read_excel(r'C:\Users\KS\Desktop\ชื่อเค4.xlsx' ,engine='calamine',usecols=['ชื่อสินค้า'])
search_terms = search_terms['ชื่อสินค้า'].dropna().tolist()

In [164]:
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process


# 1. ฟังก์ชัน Clean ข้อความ (Vectorized ผ่าน Pandas)
def clean_series(series):
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.strip()
    )


# Clean targets
cleaned_targets = clean_series(pd.Series(search_terms)).tolist()
cleaned_targets = [t for t in cleaned_targets if t]

# 2. ดึงเฉพาะ product_id ที่ "ไม่ซ้ำ (Unique)" มา Clean และ Match
unique_products = df["product_id"].drop_duplicates()
cleaned_unique_products = clean_series(unique_products)

# สร้าง Series ลิงก์ระหว่าง product_id เดิม กับ ค่าที่ clean แล้ว
product_clean_map = pd.Series(
    cleaned_unique_products.values, index=unique_products
)

# 3. คำนวณ Similarity Matrix เฉพาะรายการที่ Unique
if cleaned_targets and not cleaned_unique_products.empty:
    # เปลี่ยน scorer เป็น fuzz.QRatio เพื่อ C-Speed เต็มสปีด
    similarity_matrix = process.cdist(
        cleaned_unique_products.tolist(),
        cleaned_targets,
        scorer=fuzz.QRatio,  # ⚡ เร็วกว่า WRatio หลายเท่า
        workers=-1,
    )

    # ดึงค่า Max Similarity ของแต่ละ Unique Product
    max_scores = similarity_matrix.max(axis=1)

    # Map คะแนนกลับไปยัง Unique Product ID
    score_map = pd.Series(max_scores, index=unique_products)

    # 4. Map คะแนนกลับเข้า DataFrame หลัก
    df["max_similarity"] = df["product_id"].map(score_map).fillna(0.0)
else:
    df["max_similarity"] = 0.0

# 5. กรองเฉพาะรายการที่คะแนนถึงเกณฑ์
threshold = 90
filtered_df = df[df["max_similarity"] >= threshold].copy()

In [165]:
filtered_df[['DATE', 'Bill', 'details', 'product_id', 'import', 'export', 'balances','mode_import_size','median_import_size','Expected_Import','is_outlier_import','Anomaly_Type']].to_excel(
    r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')